# Week 66, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
free_at = [0, 0]
for d in [4, 1, 1, 3]:
    i = free_at.index(min(free_at))
    free_at[i] += d
print(free_at, max(free_at))

**Predict before you run:** what will this print?



In [ ]:
def speedup(p, n):
    return round(1 / ((1 - p) + p / n), 2)
print(speedup(0.5, 2), speedup(0.5, 1000), speedup(0.75, 3))

**Predict before you run:** what will this print?



In [ ]:
import threading
lock = threading.Lock()
with lock:
    print(lock.locked(), lock.acquire(blocking=False))
print(lock.locked())

**Predict before you run:** what will this print?



In [ ]:
from concurrent.futures import Future
f = Future()
f.add_done_callback(lambda fut: print("done:", fut.result()))
print(f.done())
f.set_result("page")

**Predict before you run:** what will this print?



In [ ]:
x, regs = 0, {}
for who, op in [("A", "load"), ("B", "load"), ("A", "store"), ("B", "store")]:
    if op == "load":
        regs[who] = x
    else:
        x = regs[who] + 1
print(x)

### Exercise: A cache shared between threads

Write a decorator `locked_cache(func)` (Week 65's caching) that memoises by positional arguments and is safe to call from many threads at once: the check **and** the store happen inside one `threading.Lock`, so two threads can never both compute the same missing entry. The wrapper has `calls` (real calls made) and `lock` (the lock), and keeps the function's name. An exception from `func` must not leave the lock held.

In [ ]:
import functools
import threading

def locked_cache(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_locked_cache():\n    \"caching, safely\"\n    @locked_cache\n    def slow_square(x):\n        \"Square a number slowly.\"\n        return x * x\n    assert slow_square(4) == 16 and slow_square(4) == 16 and slow_square(5) == 25 and slow_square.calls == 2\n    assert slow_square.__name__ == \"slow_square\" and not slow_square.lock.locked()\n    @locked_cache\n    def boom(x):\n        raise ValueError(x)\n    try:\n        boom(1)\n        assert False\n    except ValueError:\n        pass\n    assert not boom.lock.locked()")

### Exercise: What can run at the same time?

Build jobs have dependencies: `deps` maps each task to the tasks it needs first (tasks only mentioned as dependencies have none). Write `parallel_batches(deps)`: a list of batches, each a sorted list of the tasks whose dependencies are all in earlier batches, so every task in a batch could run in parallel (Week 50's topological sort, level by level), raising `ValueError("cycle")` if some tasks can never run. Then `shortest_time(deps, durations)`: the time to finish everything with unlimited workers, where each task starts as soon as its dependencies have finished.

In [ ]:
def parallel_batches(deps):
    ___

def shortest_time(deps, durations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_batches():\n    \"dependencies and parallelism\"\n    deps = {\"test\": [\"build\"], \"build\": [\"fetch\", \"configure\"], \"docs\": [\"configure\"], \"ship\": [\"test\", \"docs\"]}\n    assert parallel_batches(deps) == [[\"configure\", \"fetch\"], [\"build\", \"docs\"], [\"test\"], [\"ship\"]]\n    assert parallel_batches({}) == [] and parallel_batches({\"a\": []}) == [[\"a\"]]\n    try:\n        parallel_batches({\"a\": [\"b\"], \"b\": [\"a\"]})\n        assert False\n    except ValueError:\n        pass\n    durations = {\"fetch\": 3, \"configure\": 1, \"build\": 5, \"docs\": 2, \"test\": 4, \"ship\": 1}\n    assert shortest_time(deps, durations) == 13")

### Exercise: Forge: a crawler

Write `crawl` to the specification above, using a heap of `(finish_time, start_sequence, url)` events.

In [ ]:
import heapq
from collections import deque

def crawl(site, start, workers=2, per_host=1, max_pages=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic_crawl():\n    \"a pool of workers\"\n    site = {\n        \"a.org/\": (2, [\"a.org/x\", \"b.org/\", \"a.org/y\"]),\n        \"a.org/x\": (3, [\"a.org/\"]),\n        \"a.org/y\": (1, []),\n        \"b.org/\": (1, [\"b.org/z\", \"a.org/x\"]),\n        \"b.org/z\": (5, []),\n    }\n    one = crawl(site, \"a.org/\", workers=1, per_host=5)\n    assert one[\"order\"] == [\"a.org/\", \"a.org/x\", \"b.org/\", \"a.org/y\", \"b.org/z\"] and one[\"time\"] == 12\n    many = crawl(site, \"a.org/\", workers=3, per_host=5)\n    assert many[\"order\"] == [\"a.org/\", \"b.org/\", \"a.org/y\", \"a.org/x\", \"b.org/z\"]\n    assert many[\"finished_at\"] == {\"a.org/\": 2, \"b.org/\": 3, \"a.org/y\": 3, \"a.org/x\": 5, \"b.org/z\": 8}\n    assert many[\"time\"] == 8 and many[\"errors\"] == []\n\ndef test_limits_and_errors():\n    \"per-host limits, missing pages and page limits\"\n    site = {\n        \"s.com/\": (1, [\"s.com/1\", \"s.com/2\", \"s.com/3\", \"t.com/\"]),\n        \"s.com/1\": (4, []), \"s.com/2\": (4, []), \"s.com/3\": (4, [\"s.com/gone\"]),\n        \"t.com/\": (1, []),\n    }\n    polite = crawl(site, \"s.com/\", workers=4, per_host=1)\n    assert polite[\"finished_at\"][\"t.com/\"] == 2 and polite[\"finished_at\"][\"s.com/3\"] == 13\n    assert polite[\"errors\"] == [\"s.com/gone\"] and polite[\"time\"] == 14\n    fast = crawl(site, \"s.com/\", workers=4, per_host=3)\n    assert fast[\"finished_at\"][\"s.com/3\"] == 5 and fast[\"time\"] == 6\n    capped = crawl(site, \"s.com/\", workers=2, per_host=2, max_pages=3)\n    assert len(capped[\"order\"]) == 3 and capped[\"order\"][0] == \"s.com/\"\n    assert crawl({}, \"x.net/\", workers=1)[\"errors\"] == [\"x.net/\"]")

> **The same design with real threads.** Each part of the simulation has a real counterpart. The work queue is a `queue.Queue`, the workers are threads in a `ThreadPoolExecutor`, the per-host limit is a `Semaphore` per host, and the seen set is shared, so adding to it must happen under a `Lock` (check-then-act: "if not seen, mark seen and queue it" must be atomic, or two workers can both queue the same page). Fetching is I/O-bound, so the GIL doesn't get in the way: while one thread waits for a reply, others run. What the simulation adds is determinism: with real threads, the order of finishing would change from run to run, and a test that passed today could fail tomorrow. Designing so that results don't depend on that order is half of concurrent programming.

### Exercise: Forge bonus: rate limiting

A polite crawler limits its **rate** as well as its concurrency. Write two thread-safe classes, each protecting its state with a real `threading` primitive:

- `TokenBucket(rate, capacity, clock=time.monotonic)`: holds up to `capacity` tokens, starting full, and gains `rate` tokens per second (by the clock), never above capacity. `try_acquire(n=1)` takes `n` tokens if there are enough and returns `True`, otherwise returns `False` (raising `ValueError` if `n` is more than the capacity); `tokens()` is the current number (rounded to 6 places); `wait_time(n=1)` is how long until `n` tokens would be available (0 if they are now, rounded to 6 places); `granted` and `refused` count the results of `try_acquire`. All state changes happen under one `threading.Lock`.
- `Limiter(max_running)`: a context manager allowing at most `max_running` blocks to be inside it at once, using a `threading.BoundedSemaphore` acquired **without waiting** (raising `RuntimeError("too many running")` if none is free), with `running` and `peak` counters updated under a lock, and the slot always released when the block ends.

In [ ]:
import threading
import time

class TokenBucket:
    def __init__(self, rate, capacity, clock=time.monotonic):
        ___

class Limiter:
    def __init__(self, max_running):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_bucket():\n    \"a token bucket\"\n    now = [0.0]\n    b = TokenBucket(rate=2, capacity=4, clock=lambda: now[0])\n    assert [b.try_acquire() for _ in range(5)] == [True, True, True, True, False]\n    assert b.tokens() == 0 and b.wait_time() == 0.5 and b.wait_time(3) == 1.5\n    now[0] = 1.0\n    assert b.tokens() == 2 and b.try_acquire(2) and not b.try_acquire()\n    now[0] = 100.0\n    assert b.tokens() == 4 and b.wait_time(4) == 0.0\n    assert (b.granted, b.refused) == (5, 2)\n    try:\n        b.try_acquire(5)\n        assert False\n    except ValueError:\n        pass\n    assert not b._lock.locked()\n\ndef test_limiter():\n    \"a concurrency limit\"\n    lim = Limiter(2)\n    with lim:\n        with lim:\n            assert lim.running == 2\n            try:\n                with lim:\n                    pass\n                assert False\n            except RuntimeError as e:\n                assert str(e) == \"too many running\"\n            assert lim.running == 2\n        assert lim.running == 1\n    assert lim.running == 0 and lim.peak == 2\n    try:\n        with lim:\n            raise KeyError\n    except KeyError:\n        pass\n    assert lim.running == 0")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [processes and threads](w066d1.html), [threading](w066d2.html), [the GIL](w066d3.html), [race conditions and locks](w066d4.html) or [concurrent.futures](w066d5.html).

## Mixed review

## Forge: a crawler

A **site** is a dict from URL (like `"a.org/shop"`, whose **host** is the part before the first `/`) to `(latency, links)`: fetching it takes `latency` time units and finds those links. URLs not in the dict are missing: fetching one takes 1 unit and fails. Write `crawl(site, start, workers=2, per_host=1, max_pages=None)`:

- The crawler keeps a **queue** of URLs to fetch, in the order they were discovered, starting with `start`, and a **seen** set so that no URL is queued twice.
- At most `workers` fetches run at once, and at most `per_host` of them for the same host. Whenever a worker is free, it takes the **first** URL in the queue whose host is below its limit (others wait their turn). At most `max_pages` fetches are started in total, if given.
- When a fetch finishes, its links are added to the queue in order (skipping seen ones), or the URL is recorded as an error if it was missing. Then free workers start new fetches.
- Fetches finishing at the same time are handled in the order they started.

Return a dict with `"order"` (URLs in the order their fetches finished), `"finished_at"` (URL to finish time), `"errors"` (missing URLs, in the order they finished) and `"time"` (when the last fetch finished).

## Forge bonus: rate limiting

## Weekly quiz

## Week 66 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can tell concurrency from parallelism, threads from processes, and I/O-bound from CPU-bound work, and estimate speed-ups. ([Day 1](w066d1.html))
- I can use Thread, queue.Queue, Event and daemon threads, and simulate threads with generators. ([Day 2](w066d2.html))
- I can explain the GIL, the switch interval, and the ways around it. ([Day 3](w066d3.html))
- I can find races by enumerating interleavings, protect data with locks and prevent deadlock. ([Day 4](w066d4.html))
- I can use executors, futures, as_completed and callbacks, and write my own executor. ([Day 5](w066d5.html))
- I built a crawler simulation and thread-safe rate limiting. (Today)

Tomorrow is a **rest day**. Next week: **Concurrency II**: `multiprocessing`, `asyncio`, async I/O patterns, async pitfalls, and choosing a concurrency model.